In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [5]:
class PositionalEncoding(nn.Module):
    """
    Add positional encoding to embeddings.

    Since transformers don't have inherent notion of sequence order (unlike RNNs),
    we need to inject positional information. This uses sinusoidal patterns
    that allow the model to learn relative positions.
    """
    def __init__(self, d_model, max_length=5000):
        super().__init__()

        # Create a matrix to hold positional encodings
        # Shape: (max_length, d_model)
        pe = torch.zeros(max_length, d_model)

        # Create position indices: [0, 1, 2, ..., max_length-1]
        # Shape: (max_length, 1)
        position = torch.arange(0, max_length).unsqueeze(1).float()

        # Create the divisor term for the sinusoidal pattern
        # This creates different frequencies for different dimensions
        # Formula: 1 / (10000^(2i/d_model)) for i in [0, 2, 4, ..., d_model-2]
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                           -(math.log(10000.0) / d_model))

        # Apply sine to even indices (0, 2, 4, ...)
        pe[:, 0::2] = torch.sin(position * div_term)
        # Apply cosine to odd indices (1, 3, 5, ...)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Register as buffer so it's saved with the model but not trained
        # Add batch dimension: (1, max_length, d_model)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        """
        Add positional encoding to input embeddings.
        Args:
            x: Input embeddings of shape (batch_size, seq_len, d_model)
        Returns:
            x + positional_encoding of same shape
        """
        # Add positional encoding to input, only for the sequence length we need
        return x + self.pe[:, :x.size(1)]

In [6]:
class MultiHeadAttention(nn.Module):
    """
    Multi-head self-attention mechanism - the core of the transformer.

    This allows the model to jointly attend to information from different
    representation subspaces. Instead of one attention function, we use
    multiple 'heads' that can focus on different aspects of the input.
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        # Ensure d_model is divisible by n_heads for even head dimensions
        assert d_model % n_heads == 0

        self.d_model = d_model  # Model dimension (e.g., 512)
        self.n_heads = n_heads  # Number of attention heads (e.g., 8)
        self.d_k = d_model // n_heads  # Dimension per head (e.g., 64)

        # Linear transformations for Query, Key, Value, and Output
        # These project the input into Q, K, V spaces
        self.W_q = nn.Linear(d_model, d_model)  # Query projection
        self.W_k = nn.Linear(d_model, d_model)  # Key projection
        self.W_v = nn.Linear(d_model, d_model)  # Value projection
        self.W_o = nn.Linear(d_model, d_model)  # Output projection

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Compute scaled dot-product attention.

        Attention(Q,K,V) = softmax(QK^T / sqrt(d_k))V

        Args:
            Q: Queries of shape (batch, n_heads, seq_len, d_k)
            K: Keys of shape (batch, n_heads, seq_len, d_k)
            V: Values of shape (batch, n_heads, seq_len, d_k)
            mask: Optional mask to prevent attention to certain positions
        """
        # Step 1: Compute attention scores
        # QK^T gives us similarity between each query and key
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # Shape: (batch, n_heads, seq_len, seq_len)

        # Step 2: Apply mask if provided (e.g., to ignore padding tokens)
        if mask is not None:
            # Expand mask to match the number of heads if needed
            if mask.dim() == 4 and mask.size(1) == 1:
                mask = mask.expand(-1, self.n_heads, -1, -1)
            # Set masked positions to large negative value (becomes 0 after softmax)
            scores = scores.masked_fill(mask == 0, -1e9)

        # Step 3: Apply softmax to get attention weights (probabilities)
        attention_weights = F.softmax(scores, dim=-1)

        # Step 4: Apply attention weights to values
        output = torch.matmul(attention_weights, V)

        return output, attention_weights

    def forward(self, query, key, value, mask=None):
        """
        Forward pass of multi-head attention.

        Args:
            query, key, value: Input tensors of shape (batch, seq_len, d_model)
            mask: Optional attention mask
        Returns:
            output: Attention output of shape (batch, seq_len, d_model)
        """
        batch_size = query.size(0)

        # Step 1: Linear transformations and reshape for multiple heads
        # Transform to Q, K, V and split into heads
        # (batch, seq_len, d_model) -> (batch, seq_len, n_heads, d_k) -> (batch, n_heads, seq_len, d_k)
        Q = self.W_q(query).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        # Step 2: Apply scaled dot-product attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)

        # Step 3: Concatenate heads back together
        # (batch, n_heads, seq_len, d_k) -> (batch, seq_len, n_heads, d_k) -> (batch, seq_len, d_model)
        attention_output = attention_output.transpose(1, 2).contiguous().view(
            batch_size, -1, self.d_model)

        # Step 4: Final linear transformation
        output = self.W_o(attention_output)

        return output

In [7]:
class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network.

    This is a simple 2-layer MLP applied to each position separately.
    It provides non-linearity and allows the model to process the
    attended information further.
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        # Two linear layers with ReLU activation in between
        # Typically d_ff = 4 * d_model (e.g., 2048 when d_model=512)
        self.linear1 = nn.Linear(d_model, d_ff)  # Expand dimension
        self.linear2 = nn.Linear(d_ff, d_model)  # Contract back to d_model
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        Forward pass: Linear -> ReLU -> Dropout -> Linear

        Args:
            x: Input of shape (batch, seq_len, d_model)
        Returns:
            Output of same shape
        """
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

In [8]:
class TransformerBlock(nn.Module):
    """
    Single transformer encoder block.

    This combines multi-head attention and feed-forward network with
    residual connections and layer normalization. This is the core
    building block that gets stacked to create deep transformers.
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        # Main components
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

        # Layer normalization (applied before each sub-layer in this implementation)
        self.norm1 = nn.LayerNorm(d_model)  # For attention output
        self.norm2 = nn.LayerNorm(d_model)  # For feed-forward output

        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Forward pass with residual connections.

        Architecture:
        1. x -> LayerNorm -> MultiHeadAttention -> Dropout -> Add&Norm
        2. x -> LayerNorm -> FeedForward -> Dropout -> Add&Norm

        Args:
            x: Input of shape (batch, seq_len, d_model)
            mask: Optional attention mask
        Returns:
            Output of same shape
        """
        # Self-attention with residual connection and layer norm
        # Note: This uses pre-norm (LayerNorm before attention)
        attention_output = self.attention(x, x, x, mask)  # Self-attention: Q=K=V=x
        x = self.norm1(x + self.dropout(attention_output))  # Add & Norm

        # Feed-forward with residual connection and layer norm
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))  # Add & Norm

        return x


In [9]:
class SimpleTransformer(nn.Module):
    """
    Simple transformer model for sequence-to-sequence tasks.

    This is a complete transformer that can be used for language modeling,
    text classification, or other NLP tasks. It consists of:
    1. Token embeddings + positional encoding
    2. Stack of transformer blocks
    3. Final layer normalization and output projection
    """
    def __init__(self, vocab_size, d_model=512, n_heads=8, n_layers=6,
                 d_ff=2048, max_length=5000, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # Embedding layers
        # Convert token IDs to dense vectors
        self.embedding = nn.Embedding(vocab_size, d_model)
        # Add positional information
        self.positional_encoding = PositionalEncoding(d_model, max_length)

        # Stack of transformer blocks (the main processing layers)
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        # Output layers
        self.ln_f = nn.LayerNorm(d_model)  # Final layer normalization
        self.head = nn.Linear(d_model, vocab_size)  # Project to vocabulary size

        self.dropout = nn.Dropout(dropout)

        # Initialize weights using Xavier/Glorot normal initialization
        self.apply(self._init_weights)

    def _init_weights(self, module):
        """
        Initialize model weights.

        Proper weight initialization is crucial for training stability.
        We use normal distribution with small standard deviation.
        """
        if isinstance(module, nn.Linear):
            # Initialize linear layer weights from normal distribution
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            # Initialize embedding weights from normal distribution
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def create_padding_mask(self, x, pad_token_id=0):
        """
        Create mask to ignore padding tokens during attention.

        Args:
            x: Input token IDs of shape (batch_size, seq_len)
            pad_token_id: Token ID used for padding (usually 0)
        Returns:
            mask: Boolean mask of shape (batch_size, 1, seq_len, seq_len)
                 True for valid positions, False for padding
        """
        # Create basic mask: True where tokens are not padding
        # Shape: (batch_size, seq_len)
        mask = (x != pad_token_id)

        # Expand to attention dimensions: (batch_size, 1, 1, seq_len)
        mask = mask.unsqueeze(1).unsqueeze(1)

        # Expand to full attention matrix: (batch_size, 1, seq_len, seq_len)
        mask = mask.expand(-1, -1, x.size(1), -1)

        return mask

    def forward(self, x, mask=None):
        """
        Forward pass through the transformer.

        Args:
            x: Input token IDs of shape (batch_size, seq_len)
            mask: Optional attention mask
        Returns:
            logits: Output logits of shape (batch_size, seq_len, vocab_size)
        """
        # Step 1: Token embeddings
        # Convert token IDs to dense vectors and scale by sqrt(d_model)
        # Scaling helps with training stability when combined with positional encoding
        token_embeddings = self.embedding(x) * math.sqrt(self.d_model)

        # Step 2: Add positional encoding
        # This injects position information since attention is permutation-invariant
        x = self.positional_encoding(token_embeddings)
        x = self.dropout(x)

        # Step 3: Handle masking
        if mask is None:
            # For simplicity in basic usage, we skip masking
            # In practice, you'd pass the original token IDs separately for mask creation
            mask = None

        # Step 4: Pass through transformer blocks
        # Each block applies self-attention and feed-forward processing
        for transformer_block in self.transformer_blocks:
            x = transformer_block(x, mask)

        # Step 5: Final processing
        # Apply final layer normalization
        x = self.ln_f(x)
        # Project to vocabulary size to get logits for each token
        logits = self.head(x)

        return logits


In [10]:
class SimpleTokenizer:
    """
    A simple character-level tokenizer for demonstration.
    In practice, you'd use more sophisticated tokenizers like BPE or WordPiece.
    """
    def __init__(self, texts):
        print("Building vocabulary from text data...")

        # Special tokens
        self.pad_token = '<PAD>'
        self.unk_token = '<UNK>'
        self.bos_token = '<BOS>'  # Beginning of sequence
        self.eos_token = '<EOS>'  # End of sequence

        # Build character vocabulary from all texts
        all_chars = set()
        for text in texts:
            all_chars.update(text.lower())

        # Create vocabulary: special tokens + characters
        vocab_list = [self.pad_token, self.unk_token, self.bos_token, self.eos_token]
        vocab_list.extend(sorted(all_chars))

        # Create mappings
        self.token_to_id = {token: i for i, token in enumerate(vocab_list)}
        self.id_to_token = {i: token for i, token in enumerate(vocab_list)}
        self.vocab_size = len(vocab_list)

        print(f"Vocabulary size: {self.vocab_size}")
        print(f"Sample vocabulary: {vocab_list[:20]}...")

    def encode(self, text, max_length=None, add_special_tokens=True):
        """Convert text to token IDs"""
        text = text.lower()

        if add_special_tokens:
            tokens = [self.bos_token] + list(text) + [self.eos_token]
        else:
            tokens = list(text)

        # Convert to IDs
        token_ids = [self.token_to_id.get(token, self.token_to_id[self.unk_token])
                    for token in tokens]

        # Truncate or pad
        if max_length:
            if len(token_ids) > max_length:
                token_ids = token_ids[:max_length]
            else:
                pad_id = self.token_to_id[self.pad_token]
                token_ids.extend([pad_id] * (max_length - len(token_ids)))

        return token_ids

    def decode(self, token_ids, skip_special_tokens=True):
        """Convert token IDs back to text"""
        tokens = [self.id_to_token[id] for id in token_ids]

        if skip_special_tokens:
            special_tokens = {self.pad_token, self.unk_token, self.bos_token, self.eos_token}
            tokens = [t for t in tokens if t not in special_tokens]

        return ''.join(tokens)

def create_real_world_dataset():
    """
    Create a dataset using real-world text examples.
    In practice, you'd load this from files or datasets.
    """
    # Sample texts from different domains
    texts = [
        # Literature samples
        "To be or not to be, that is the question. Whether 'tis nobler in the mind to suffer the slings and arrows of outrageous fortune.",
        "It was the best of times, it was the worst of times, it was the age of wisdom, it was the age of foolishness.",
        "In the beginning was the Word, and the Word was with God, and the Word was God.",

        # Scientific texts
        "The theory of relativity revolutionized our understanding of space and time, showing that they are interconnected.",
        "Machine learning algorithms can identify patterns in data that would be impossible for humans to detect manually.",
        "Climate change is caused by increased greenhouse gas concentrations in the atmosphere due to human activities.",

        # News/journalism
        "Breaking news: Scientists have discovered a new method for generating clean energy using renewable resources.",
        "The stock market reached new heights today as investors showed confidence in emerging technologies.",
        "Local communities are working together to address environmental challenges through innovative solutions.",

        # Conversational/social
        "Hello there! How are you doing today? I hope you're having a wonderful time learning about transformers.",
        "Thank you for your help with the project. I really appreciate all the effort you put into making it successful.",
        "Would you like to grab coffee sometime this week? I'd love to discuss our upcoming collaboration.",

        # Technical/programming
        "Neural networks consist of layers of interconnected nodes that process information through weighted connections.",
        "Python is a versatile programming language used for web development, data science, and artificial intelligence.",
        "The transformer architecture uses self-attention mechanisms to process sequential data efficiently.",

        # Creative writing
        "The old lighthouse stood silently against the storm, its beacon cutting through the darkness like hope itself.",
        "She walked through the enchanted forest, where every leaf seemed to whisper ancient secrets of forgotten magic.",
        "In the bustling city streets, millions of stories unfold simultaneously, each person carrying their own dreams."
    ]

    return texts

In [11]:
# Example usage and testing with real-world data
if __name__ == "__main__":
    print("=== Simple Transformer with Real-World Data ===\n")

    # Step 1: Create real-world dataset
    print("Creating real-world text dataset...")
    texts = create_real_world_dataset()

    print(f"Number of text samples: {len(texts)}")
    print(f"Sample text: '{texts[0][:60]}...'\n")

    # Step 2: Build tokenizer
    tokenizer = SimpleTokenizer(texts)

    # Step 3: Model hyperparameters
    seq_length = 128      # Maximum sequence length
    batch_size = 8        # Process 8 texts at once

    print(f"\nCreating transformer model with vocab_size={tokenizer.vocab_size}...")
    # Create model with vocabulary size matching our tokenizer
    model = SimpleTransformer(
        vocab_size=tokenizer.vocab_size,  # Use actual vocabulary size
        d_model=256,                      # Smaller for demo (512 is standard)
        n_heads=8,                        # Number of attention heads
        n_layers=4,                       # Fewer layers for demo (6-12 is typical)
        d_ff=1024,                        # Feed-forward hidden dimension
        dropout=0.1                       # Dropout rate
    )

    # Step 4: Tokenize real text data
    print("\nTokenizing text data...")
    tokenized_texts = []
    original_texts = []

    for i, text in enumerate(texts[:batch_size]):  # Use first batch_size texts
        token_ids = tokenizer.encode(text, max_length=seq_length)
        tokenized_texts.append(token_ids)
        original_texts.append(text)

        if i < 3:  # Show first few examples
            print(f"Text {i+1}: '{text[:50]}...'")
            print(f"Tokens: {token_ids[:20]}...")  # Show first 20 tokens
            print(f"Decoded: '{tokenizer.decode(token_ids[:20])}'")
            print()

    # Convert to tensor
    input_ids = torch.tensor(tokenized_texts)
    print(f"Input tensor shape: {input_ids.shape}")

    # Step 5: Create attention masks for padding
    print("\nCreating attention masks...")
    attention_mask = model.create_padding_mask(input_ids, pad_token_id=tokenizer.token_to_id[tokenizer.pad_token])
    print(f"Attention mask shape: {attention_mask.shape}")

    # Step 6: Forward pass with real data
    print("\nRunning forward pass with real text data...")
    with torch.no_grad():
        outputs = model(input_ids, mask=attention_mask)

    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Output shape: {outputs.shape}")
    print(f"Output logits range: [{outputs.min():.3f}, {outputs.max():.3f}]")

    # Step 7: Show predictions for first few tokens
    print("\n--- Model Predictions Analysis ---")
    with torch.no_grad():
        # Get predictions for the first sequence
        first_seq_logits = outputs[0]  # Shape: (seq_len, vocab_size)
        first_seq_probs = F.softmax(first_seq_logits, dim=-1)

        print(f"Analyzing predictions for: '{original_texts[0][:60]}...'")
        print("\nTop predictions for first few positions:")

        for pos in range(min(5, seq_length)):
            if input_ids[0, pos] == tokenizer.token_to_id[tokenizer.pad_token]:
                break

            # Get top 3 predictions for this position
            top_probs, top_indices = torch.topk(first_seq_probs[pos], 3)

            actual_token = tokenizer.id_to_token[input_ids[0, pos].item()]
            print(f"Position {pos}: Actual='{actual_token}'")

            for i, (prob, idx) in enumerate(zip(top_probs, top_indices)):
                predicted_token = tokenizer.id_to_token[idx.item()]
                print(f"  Top {i+1}: '{predicted_token}' (prob: {prob:.3f})")
            print()

    # Step 8: Training setup example
    print("\n--- Training Setup for Real Data ---")
    print("# For language modeling, create input-target pairs:")
    print("# inputs = tokenized_text[:-1]   # All tokens except last")
    print("# targets = tokenized_text[1:]   # All tokens except first")
    print("")
    print("# Training setup:")
    print("optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)")
    print("criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.token_to_id['<PAD>'])")
    print("scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)")
    print("")
    print("# Training metrics to track:")
    print("# - Perplexity: exp(cross_entropy_loss)")
    print("# - Accuracy: Percentage of correct next-token predictions")
    print("# - Learning curves: Loss over time")

    # Step 9: Show what the model would learn
    print("\n--- What the Model Learns ---")
    print("During training, this transformer would learn to:")
    print("• Predict the next character/token given previous context")
    print("• Understand patterns in language (grammar, semantics)")
    print("• Attend to relevant parts of the input sequence")
    print("• Build representations that capture meaning and structure")
    print("• Generate coherent text by sampling from learned distributions")

    print(f"\nDataset statistics:")
    print(f"• Total characters: {sum(len(text) for text in texts):,}")
    print(f"• Vocabulary size: {tokenizer.vocab_size}")
    print(f"• Average sequence length: {sum(len(tokenizer.encode(text, add_special_tokens=False)) for text in texts) // len(texts)}")
    print(f"• Model size: {sum(p.numel() for p in model.parameters()):,} parameters")

=== Simple Transformer with Real-World Data ===

Creating real-world text dataset...
Number of text samples: 18
Sample text: 'To be or not to be, that is the question. Whether 'tis noble...'

Building vocabulary from text data...
Vocabulary size: 37
Sample vocabulary: ['<PAD>', '<UNK>', '<BOS>', '<EOS>', ' ', '!', "'", ',', '-', '.', ':', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']...

Creating transformer model with vocab_size=37...

Tokenizing text data...
Text 1: 'To be or not to be, that is the question. Whether ...'
Tokens: [2, 31, 26, 4, 13, 16, 4, 26, 29, 4, 25, 26, 31, 4, 31, 26, 4, 13, 16, 7]...
Decoded: 'to be or not to be,'

Text 2: 'It was the best of times, it was the worst of time...'
Tokens: [2, 20, 31, 4, 34, 12, 30, 4, 31, 19, 16, 4, 13, 16, 30, 31, 4, 26, 17, 4]...
Decoded: 'it was the best of '

Text 3: 'In the beginning was the Word, and the Word was wi...'
Tokens: [2, 20, 25, 4, 31, 19, 16, 4, 13, 16, 18, 20, 25, 25, 20, 25, 18, 4, 34, 12]...
Decoded: 'in the begi